# Решения · handbook гл. 37

Открывай **только после** [handbook_gl37_практика.ipynb](./handbook_gl37_практика.ipynb).

Источник: [dl-regularization](https://mlhandbook-production.up.railway.app/parts/deep-learning/dl-regularization.html)


# §37.2 · Dropout train/eval

In [ ]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(100, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 10))
x = torch.randn(4, 100)
net.train()
out_train = net(x)
net.eval()
out_eval = net(x)
print("train и eval разные:", not torch.allclose(out_train, out_eval))


# §37.7 · Dropout vs overfit

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


def fit_with_dropout(use_dropout: bool):
    torch.manual_seed(0)
    X = torch.randn(48, 8)
    y = (X[:, :3].sum(dim=1) > 0).float().unsqueeze(1)
    Xt = torch.randn(120, 8)
    yt = (Xt[:, :3].sum(dim=1) > 0).float().unsqueeze(1)

    layers = [nn.Linear(8, 32), nn.ReLU()]
    if use_dropout:
        layers += [nn.Dropout(0.5)]
    layers += [nn.Linear(32, 1), nn.Sigmoid()]
    model = nn.Sequential(*layers)
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    lossf = nn.BCELoss()
    tr, te = [], []
    for _ in range(80):
        model.train()
        opt.zero_grad()
        lt = lossf(model(X), y)
        lt.backward()
        opt.step()
        tr.append(float(lt.item()))
        model.eval()
        with torch.no_grad():
            te.append(float(lossf(model(Xt), yt).item()))
    return tr, te


tr0, te0 = fit_with_dropout(False)
tr1, te1 = fit_with_dropout(True)
plt.plot(te0, label="test без dropout")
plt.plot(te1, label="test с dropout")
plt.legend()
plt.ylabel("loss")
plt.show()


# Lab · RegularizedMLP

In [ ]:
import torch
from torch import nn


class RegularizedMLP(nn.Module):
    def __init__(self, in_dim=20, hidden=32, out_dim=2, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(0)
X = torch.randn(64, 20)
y = torch.randint(0, 2, (64,))
model = RegularizedMLP()
opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
lossf = nn.CrossEntropyLoss()
for _ in range(30):
    model.train()
    opt.zero_grad()
    lossf(model(X), y).backward()
    opt.step()
model.eval()
with torch.no_grad():
    acc = (model(X).argmax(1) == y).float().mean()
print(model)
print("train acc (demo):", float(acc))


# Сверка NumPy ↔ torch

In [ ]:
import numpy as np
import torch
from torch import nn


def need(cond, msg=""):
    assert cond, msg


def dropout_numpy(x, p=0.5, training=True):
    if not training:
        return x
    mask = (np.random.rand(*x.shape) > p).astype(float)
    return x * mask / (1.0 - p)


np.random.seed(0)
x = np.ones((32, 8))
y = dropout_numpy(x, training=True)
need(abs(x.mean() - y.mean()) < 0.2)

torch.manual_seed(0)
bn = nn.BatchNorm1d(8)
bn.train()
for _ in range(5):
    _ = bn(torch.randn(16, 8))
bn.eval()
xt = torch.randn(4, 8)
torch_out = bn(xt)
mu = bn.running_mean
var = bn.running_var
manual = (xt - mu) / torch.sqrt(var + bn.eps) * bn.weight + bn.bias
need(torch.allclose(torch_out, manual, atol=1e-5))
print("dropout mean ok, BN eval ok")
